# Week 4: UMAP for top-3 models vs Week 3

Global cosine UMAP on balanced AE pixels. Claim-first figures (not a 4×10 pred grid): GT + 2D LR boundary, F1/agreement by metro, error spotlight.

Top 3: W4 balanced LR, W3 Ridge, W3 AE LR.

Rebuild: `.venv/bin/python -u scripts/plot_week4_top3_umap.py`


In [1]:
from pathlib import Path
import json
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = HERE
for p in [HERE, *HERE.parents]:
    if (p / "outputs" / "week4_top3_umap" / "summary.json").exists():
        ROOT = p
        break

OUT = ROOT / "outputs" / "week4_top3_umap"
summary = json.loads((OUT / "summary.json").read_text())
gt_f1 = pd.read_csv(OUT / "in_umap_lr_f1_gt.csv")
model_f1 = pd.read_csv(OUT / "in_umap_lr_f1_by_model.csv")
per = pd.read_csv(OUT / "per_region_f1.csv")

print(f"n={summary['umap']['n_pixels']:,}  mean in-UMAP LR F1 (GT)={summary.get('in_umap_lr_f1_gt_mean'):.3f}")
display(pd.DataFrame(summary["models"]).round(4))
display(gt_f1.sort_values("in_umap_lr_f1", ascending=False).round(3))
pivot = model_f1.pivot(index="region", columns="model", values="in_umap_lr_f1")
display(pivot.round(3))


n=59,596  mean in-UMAP LR F1 (GT)=0.692


,model,label,n,macro_f1_on_umap_set,accuracy
0,balanced_train,balanced_train (W4),59596,0.7043,0.7051
1,fraction_regression,fraction_regression (W3),59596,0.6999,0.7000
2,pixel_ae,pixel_ae (W3),59596,0.6972,0.6976


,region,n,in_umap_lr_f1,y_col
7,Phoenix_SW,5726,0.801,y
5,LA_W,3088,0.750,y
9,Seattle_NW,6496,0.711,y
2,Chicago_MW,6136,0.702,y
8,SF_W,2822,0.701,y
0,Atlanta_SE,9340,0.695,y
3,Denver_W,5502,0.674,y
4,Houston_S,8264,0.668,y
1,Boston_NE,6688,0.661,y
6,NYC_NE,5534,0.555,y


model,balanced_train,fraction_regression,pixel_ae
region,,,
Atlanta_SE,0.750,0.772,0.746
Boston_NE,0.712,0.733,0.731
Chicago_MW,0.752,0.771,0.738
Denver_W,0.838,0.828,0.810
Houston_S,0.713,0.716,0.693
LA_W,0.782,0.820,0.791
NYC_NE,0.584,0.592,0.589
Phoenix_SW,0.769,0.783,0.761
SF_W,0.828,0.865,0.841


## GT: residential vs non-residential + 2D LR boundary

Blue = residential, orange = non-residential (colorblind-safe). Soft wash = P(residential) from a 2D LR in the UMAP plane. Curve = `P=0.5`. Panel titles show in-UMAP macro-F1.

Mean in-UMAP LR F1 ≈ 0.69 (Phoenix 0.80, NYC 0.56).

![GT boundary](../../outputs/week4_top3_umap/figures/umap_gt_by_region_boundary.png)

Phoenix separates cleanly; NYC collapses. That is within-metro overlap, not a missing overall signal.


## Top-3 predictions, same boundary style

Points colored by model prediction; 2D LR boundary fit on those predictions (how separable the model's RES/NRES call is in UMAP).

### balanced_train (W4)
![W4 boundary](../../outputs/week4_top3_umap/figures/umap_pred_balanced_train_by_region_boundary.png)

Week 4 balanced LR draws compact metro-wise cuts (mean in-UMAP F1 0.75) that track GT islands closely.

### fraction_regression (W3)
![Ridge boundary](../../outputs/week4_top3_umap/figures/umap_pred_fraction_regression_by_region_boundary.png)

Ridge majority calls are the most linearly tidy in UMAP (mean 0.77). Soft targets plus SMOTE give slightly cleaner 2D clusters than hard LR.

### pixel_ae (W3)
![AE boundary](../../outputs/week4_top3_umap/figures/umap_pred_pixel_ae_by_region_boundary.png)

Binary AE LR matches Week 4 geometry almost panel-for-panel (mean 0.74). Differences are edge pixels, not whole cities.


## Agreement and F1 by metro (prefer over a pred grid)

All three models match GT on ~71% of balanced pixels; differences are small. Sorted bars encode the claim directly.

![Agreement vs GT](../../outputs/week4_top3_umap/figures/agreement_vs_gt_by_metro.png)

![F1 by region](../../outputs/week4_top3_umap/figures/f1_by_region_top3.png)

Chicago and LA lead (~0.74–0.78); Houston and Atlanta lag (~0.65–0.68). Week 4 only narrowly wins most bars.


## Where W4 is wrong

Lowest / median / highest W4 F1 metros. Grey hexbin = all pixels; orange = W4 mistakes. Errors concentrate where RES and NRES mix (Atlanta is mixed throughout; Chicago mistakes sit on the NRES lobe).

![Error spotlight](../../outputs/week4_top3_umap/figures/umap_errors_spotlight.png)


## Takeaways

1. Within metros, RES and NRES partly separate in AE UMAP (mean in-UMAP LR F1 ≈ 0.69).
2. Top-3 heads agree with GT on ~71% of balanced pixels; F1 gaps across models are small.
3. W4 errors sit in mixed RES/NRES regions, not uniform noise across the embedding.
4. Metros still structure the embedding, so LORO remains the right protocol.

